# 🚀 Training Supply Chain AI/ML Models in Colab
This notebook is designed to train the PyTorch Deep Learning (LSTM) models and the Scikit-Learn (Random Forest) tabular models for your `Clean-SupplyChain-Finance` backend directly on Google Colab GPU hardware.

### Step 1: Mount Google Drive
We mount Google Drive to permanently export our `.pt` and `.joblib` model weight files so you can download them into your Java Spring Boot application.

In [ ]:
from google.colab import drive
import os

# Mount GDrive
drive.mount('/content/drive')

# Create the export directory
EXPORT_DIR = '/content/drive/MyDrive/SupplyChain_Models'
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"Models will be saved to: {EXPORT_DIR}")

### Step 2: Install Advanced Deep Learning Requirements
Since Colab comes with PyTorch pre-installed, we just need to ensure our structural dependencies are met.

In [ ]:
!pip install scikit-learn pandas numpy joblib xgboost

### Step 3: Train Scikit-Learn Random Forest Models
These models are incredibly fast. We will generate simulated production data for our Fraud Detection Classifier and Risk Assessment Classifier.

In [ ]:
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

print("Training Scikit-Learn Tabular Models...")
np.random.seed(42)

# 1. Train Demand Forecast Model (Regressor)
X_demand = np.random.randint(0, 1000, size=(10000, 4))
y_demand = X_demand[:,0] * 0.5 + X_demand[:,1] * 2 + np.random.normal(0, 10, 10000)
demand_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
demand_model.fit(X_demand, y_demand)
joblib.dump(demand_model, f"{EXPORT_DIR}/demand_forecast_model.joblib")
print("\u2705 Demand Forest Saved!")

# 2. Train Fraud Detection Model (Classifier)
X_fraud = np.random.rand(10000, 4) * 1000
y_fraud = (X_fraud[:, 1] > 800).astype(int) # High amount = fraud proxy
fraud_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
fraud_model.fit(X_fraud, y_fraud)
joblib.dump(fraud_model, f"{EXPORT_DIR}/fraud_detection_model.joblib")
print("\u2705 Fraud Forest Saved!")

# 3. Train Risk Assessment Model (Classifier)
X_risk = np.random.rand(10000, 4)
y_risk = (X_risk[:, 0] + X_risk[:, 1] > 1.0).astype(int)
risk_model = RandomForestClassifier(n_estimators=100, probability=True, n_jobs=-1)
risk_model.fit(X_risk, y_risk)
joblib.dump(risk_model, f"{EXPORT_DIR}/risk_assessment_model.joblib")
print("\u2705 Risk Assessment Forest Saved!")

### Step 4: Define and Train PyTorch LSTM Model
Demand Forecasting leverages a time-series LSTM network. We will train it using GPU acceleration.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMModel(nn.Module):
    def __init__(self, n_features=10, hidden_dim=50, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.dense = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Take the output of the last time step
        last_step_out = lstm_out[:, -1, :]
        return self.dense(last_step_out)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
SEQUENCE_LENGTH = 30
N_FEATURES = 10
EPOCHS = 50
BATCH_SIZE = 64

model = LSTMModel(n_features=N_FEATURES).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Generate Fake Tensor Data (In production, load from a parquet/CSV!)
X_train = torch.rand((5000, SEQUENCE_LENGTH, N_FEATURES), dtype=torch.float32).to(device)
y_train = torch.rand((5000, 1), dtype=torch.float32).to(device)

print("Beginning PyTorch Training Loop...")
for epoch in range(EPOCHS):
    model.train()
    # Simplified full-batch training for demonstration
    # Ensure you write a DataLoader loop here if your RAM limit hits
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {loss.item():.4f}')

# Save PyTorch state dictionary
torch.save(model.state_dict(), f"{EXPORT_DIR}/demand_forecast_dl_model.pt")
print(f"\u2705 PyTorch Demand Model Saved to {EXPORT_DIR}/demand_forecast_dl_model.pt")

### Step 5: Exporting into the Java Runtime
Check your Google Drive! A folder called `SupplyChain_Models` has been created.

1. Download the `.pt` and `.joblib` files to your computer.
2. Copy them into the `Clean-SupplyChain-Finance/backend/ai-ml/models/` directory.
3. Start your Spring Boot backend! The Java AI Service will automatically detect the weights and spin up immediate inference!